In [1]:
#data manipulation
import pandas as pd
import numpy as np

#viz
import seaborn as sns
import matplotlib.pyplot as plt

#NLP
import re
import nltk
import string
from nltk import sent_tokenize, word_tokenize
from nltk.tokenize.toktok import ToktokTokenizer
from bs4 import BeautifulSoup
import spacy
from nltk.stem import WordNetLemmatizer

%matplotlib inline

#MISC
import warnings
warnings.filterwarnings('ignore')
import unicodedata
from tqdm import tqdm

In [2]:
#Reading  data files
df = pd.read_csv(r"train.csv")

In [3]:
TAG_RE = re.compile(r'<[^>]+>') # remove <br> and other html tags

def clean_text(text):
    # lower text
    text = text.lower()

    # Reemove Html Tags like <br>
    text= TAG_RE.sub('', text)

    # remove puncutation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize_text
    text = [word for word in word_tokenize(text)]

    # remove words that contain numbers
    text = [word for word in text if not any(c.isdigit() for c in word)]
    
    # remove empty tokens
    text = [t for t in text if len(t) > 0]

    # remove words with only one letter
    text = [t for t in text if len(t) > 1]

    # join all
    text = " ".join(text)
    return(text)

In [4]:
df["comments_clean"] = df["comment_text"].apply(lambda x: clean_text(x))

In [5]:
df = df.replace(r'^(\s?)+$',np.nan,regex=True)
df = df[df['comments_clean'].notna()]
df.reset_index(inplace= True)

In [6]:
# Let's create another column "bad_comment" denoting any type of toxicity in the comment

df["bad_comment"] = df.iloc[:,2:].sum(axis = 1)
df["bad_comment"][df["bad_comment"] != 0] = 1

In [7]:
from sklearn.model_selection import train_test_split

train_corpus, test_corpus, train_toxic, test_toxic, train_severe_toxic, test_severe_toxic, train_obscene, test_obscene, train_threat, test_threat,train_insult, test_insult, train_identity_hate, test_identity_hate, train_bad_comment, test_bad_comment= train_test_split(np.array(df['comments_clean']),
                                       np.array(df['toxic']),np.array(df['severe_toxic']),np.array(df['obscene']),np.array(df['threat']),np.array(df['insult']),np.array(df['identity_hate']),np.array(df['bad_comment']),test_size=0.3, random_state=42)

# Word2Vec

In [8]:
from gensim.models import Word2Vec
Test_Bigger_list = []
for i  in tqdm(train_corpus):
    Test_li = list(i.split(" "))
    Test_Bigger_list.append(Test_li)


100%|███████████████████████████████████████████████████████████████████████| 111670/111670 [00:01<00:00, 69244.75it/s]


In [9]:
Test_Model = Word2Vec(Test_Bigger_list,min_count=1,size=5000,sg = 0) # What is sg?

In [10]:
import time

# Function to average all word vectors in a paragraph
def featureVecMethod(words, model, num_features):
    # Pre-initialising empty numpy array for speed
    featureVec = np.zeros(num_features,dtype="float32")
    nwords = 0
    
    #Converting Index2Word which is a list to a set for better speed in the execution.
    index2word_set = set(model.wv.index2word)
    
    for word in  words:
        if word in index2word_set:
            nwords = nwords + 1
            featureVec = np.add(featureVec,model[word])
    
    # Dividing the result by number of words to get average
    featureVec = np.divide(featureVec, nwords)
    return featureVec

In [11]:
# Function for calculating the average feature vector
def getAvgFeatureVecs(reviews, model, num_features):
    counter = 0
    reviewFeatureVecs = np.zeros((len(reviews),num_features),dtype="float32")
    for review in reviews:
        # Printing a status message every 1000th review
        if counter%1000 == 0:
            print("Review %d of %d"%(counter,len(reviews)))
            
        reviewFeatureVecs[counter] = featureVecMethod(review, model, num_features)
        counter = counter+1
        
    return reviewFeatureVecs

In [12]:
%%time
# Calculating average feature vector for training set
Test_clean_train_comments = []
for comment in train_corpus:
    Test_clean_train_comments.append(comment.split())
    
Test_trainDataVecs = getAvgFeatureVecs(Test_clean_train_comments, Test_Model, 5000)

Review 0 of 111670


KeyboardInterrupt: 

In [ ]:
%%time
# Calculating average feature vactors for test set     
Test_clean_test_comments = []
for comment in test_corpus:
    Test_clean_test_comments.append(comment.split())
    
Test_testDataVecs = getAvgFeatureVecs(Test_clean_test_comments, Test_Model, 5000)

Review 0 of 47859
Review 1000 of 47859
Review 2000 of 47859
Review 3000 of 47859
Review 4000 of 47859
Review 5000 of 47859


In [ ]:
drop_index = []
for i  in range(len(Test_testDataVecs)):
    for word in Test_testDataVecs[i]:
        if word != word:
            drop_index.append(i)
            print(i)
            break;

drop_index = np.array(drop_index)
drop_index = sorted(drop_index,reverse= True)
drop_index

Dropped_Test_testDataVecs = Test_testDataVecs.copy()
for i in tqdm(drop_index):
    Dropped_Test_testDataVecs = np.delete(Dropped_Test_testDataVecs,i,axis=0)
len(Dropped_Test_testDataVecs)

Dropped_test_bad_comment = test_bad_comment.copy()
for i in tqdm(drop_index):
    Dropped_test_bad_comment = np.delete(Dropped_test_bad_comment,i,axis = 0)
len(Dropped_test_bad_comment)

In [ ]:
%%time
from sklearn.metrics import confusion_matrix

# Logistic Regression
from sklearn.linear_model import LogisticRegression
Test_lr_wv = LogisticRegression(random_state = 0)
Test_lr_wv.fit(Test_trainDataVecs,train_bad_comment)

In [ ]:
%%time
Test_lr_wv_pred = Test_lr_wv.predict(Dropped_Test_testDataVecs)

In [ ]:
%%time
# Plotting Confusion Matrix
cf_matrix = confusion_matrix(Dropped_test_bad_comment,Test_lr_wv_pred)

grods_names = ['True Neg','False Pos','False Neg','True Pos']

grods_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

grods_percentages = ["{0:.2%}".format(value) for value in cf_matrix.flatten()/np.sum(cf_matrix)]

labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in zip(grods_names,grods_counts,grods_percentages)]

labels = np.asarray(labels).reshape(2,2)

plt.figure(figsize=(6,6))
ax = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap='Blues')
bottom, top = ax.get_ylim()
ax.set_ylim(bottom+0.5,top-0.5)
plt.show()

accuracy= np.trace(cf_matrix)/float(np.sum(cf_matrix))
precision= cf_matrix[1,1]/sum(cf_matrix[:,1])
recall = cf_matrix[1,1]  /  sum(cf_matrix[1,:])
f1_score  = 2*precision*recall / (precision + recall)
stats_text = "\n\nAccuracy={:0.3f}\nPrecision={:0.3f}\nRecall={:0.3f}\nF1 Score={:0.3f}".format(
accuracy,precision,recall,f1_score)
print(stats_text)

In [ ]:
%%time
# Naive Bayes
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB(alpha=1)
mnb.fit(Test_trainDataVecs,train_bad_comment)
mnb_wv_pred = mnb.predict(Dropped_Test_testDataVecs)

# Plotting Confusion Matrix
cf_matrix = confusion_matrix(Dropped_test_bad_comment,mnb_wv_pred)

grods_names = ['True Neg','False Pos','False Neg','True Pos']

grods_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

grods_percentages = ["{0:.2%}".format(value) for value in cf_matrix.flatten()/np.sum(cf_matrix)]

labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in zip(grods_names,grods_counts,grods_percentages)]

labels = np.asarray(labels).reshape(2,2)

plt.figure(figsize=(6,6))
ax = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap='Blues')
bottom, top = ax.get_ylim()
ax.set_ylim(bottom+0.5,top-0.5)
plt.show()

accuracy= np.trace(cf_matrix)/float(np.sum(cf_matrix))
precision= cf_matrix[1,1]/sum(cf_matrix[:,1])
recall = cf_matrix[1,1]  /  sum(cf_matrix[1,:])
f1_score  = 2*precision*recall / (precision + recall)
stats_text = "\n\nAccuracy={:0.3f}\nPrecision={:0.3f}\nRecall={:0.3f}\nF1 Score={:0.3f}".format(
accuracy,precision,recall,f1_score)
print(stats_text)

In [ ]:
%%time
# SVM
from sklearn.svm import LinearSVC

svm = LinearSVC(penalty='l2', C=1, random_state=42)
svm.fit(Test_trainDataVecs,train_bad_comment)
svm_wv_pred =svm.predict(Dropped_Test_testDataVecs)

# Plotting Confusion Matrix
cf_matrix = confusion_matrix(Dropped_test_bad_comment,svm_wv_pred)

grods_names = ['True Neg','False Pos','False Neg','True Pos']

grods_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

grods_percentages = ["{0:.2%}".format(value) for value in cf_matrix.flatten()/np.sum(cf_matrix)]

labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in zip(grods_names,grods_counts,grods_percentages)]

labels = np.asarray(labels).reshape(2,2)

plt.figure(figsize=(6,6))
ax = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap='Blues')
bottom, top = ax.get_ylim()
ax.set_ylim(bottom+0.5,top-0.5)
plt.show()

accuracy= np.trace(cf_matrix)/float(np.sum(cf_matrix))
precision= cf_matrix[1,1]/sum(cf_matrix[:,1])
recall = cf_matrix[1,1]  /  sum(cf_matrix[1,:])
f1_score  = 2*precision*recall / (precision + recall)
stats_text = "\n\nAccuracy={:0.3f}\nPrecision={:0.3f}\nRecall={:0.3f}\nF1 Score={:0.3f}".format(
accuracy,precision,recall,f1_score)
print(stats_text)

In [ ]:
drop_index = []
for i  in range(len(Test_testDataVecs)):
    for word in Test_testDataVecs[i]:
        if word != word:
            drop_index.append(i)
            print(i)
            break;

In [ ]:
type(drop_index)

In [ ]:
len(drop_index)

In [ ]:
drop_index = np.array(drop_index)
drop_index = sorted(drop_index,reverse= True)
drop_index

In [ ]:
Dropped_Test_testDataVecs = Test_testDataVecs.copy()


In [ ]:
for i in tqdm(drop_index):
    Dropped_Test_testDataVecs = np.delete(Dropped_Test_testDataVecs,i,axis=0)
len(Dropped_Test_testDataVecs)
    

In [ ]:
Dropped_Test_testDataVecs[:5,:2]

In [ ]:
all(np.isnan(Test_testDataVecs[2275]))

In [ ]:
test_corpus[2275]

In [ ]:
Test_clean_test_comments[2275]

In [ ]:
Test_clean_train_comments[1761]

In [ ]:
x = []
x.append(test_corpus[2275].split())
x

In [ ]:
all(np.isnan(Test_trainDataVecs[2009]))

In [ ]:
Test_clean_train_comments[2009]

In [ ]:
train_corpus[2009].split()

In [ ]:
Test_Bigger_list[2009]

In [ ]:
Test_Bigger_list[1761]